# The OLS Normal Equation

Wiki reference for [the OLS normal equation](https://ml-viz-ruby.vercel.app/wiki/ols-normal-equation).

**The idea in one sentence.** Ordinary least squares has a closed-form solution
$w^* = (X^\top X)^{-1} X^\top y$ (the normal equation) that recovers the best-fit weights in one
solve — but it becomes **numerically unstable** when features are collinear ($X^\top X$
near-singular), which is exactly why ridge regression adds $\lambda I$.

We implement OLS from scratch, **validate weight recovery and fit quality**, then cover the
gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## From-scratch OLS

In [ ]:
def ols(X, y):
    """w* = (X^T X)^{-1} X^T y  (solved via np.linalg.solve for numerical stability)"""
    return np.linalg.solve(X.T @ X, X.T @ y)

def add_bias(X):
    """Prepend a column of ones for the bias term w0."""
    return np.column_stack([np.ones(len(X)), X])

def r_squared(y, y_hat):
    ss_res = np.sum((y - y_hat)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return 1 - ss_res / ss_tot

# Wiki worked example: 3 points
X_raw = np.array([1., 2., 3.])
y     = np.array([2., 3., 5.])
X = add_bias(X_raw)

w = ols(X, y)
y_hat = X @ w
print(f"w = {w.round(3)}")          # [0.333, 1.5]
print(f"Residuals: {(y-y_hat).round(3)}")  # [ 0.167, -0.333, 0.167]
print(f"R² = {r_squared(y,y_hat):.3f}")    # 0.964
print(f"Prediction at x=4: {np.array([1,4]) @ w:.3f}")  # 6.333

## OLS vs. lstsq vs. Ridge on synthetic data

In [ ]:
rng = np.random.default_rng(42)
n, d = 50, 3
X_syn = rng.standard_normal((n, d))
w_true = np.array([1.0, -2.0, 0.5])
y_syn = X_syn @ w_true + 0.3*rng.standard_normal(n)

X_syn_b = add_bias(X_syn)

# OLS
w_ols = ols(X_syn_b, y_syn)

# lstsq (handles near-singular X^TX gracefully)
w_ls, *_ = np.linalg.lstsq(X_syn_b, y_syn, rcond=None)

print(f"True w:   {w_true}")
print(f"OLS  w:   {w_ols[1:].round(3)}")   # skip bias
print(f"lstsq w:  {w_ls[1:].round(3)}")

### Validate: OLS recovers the true weights

On well-conditioned synthetic data the normal equation recovers the generating weights (up to
noise) and matches numpy's `lstsq`. We confirm both.

In [ ]:
print(f'true {w_true},  OLS {w_ols[1:].round(3)}')
assert np.allclose(w_ols[1:], w_true, atol=0.2), 'OLS recovers the true weights (up to noise)'
assert np.allclose(w_ols, w_ls, atol=1e-8), 'the normal equation matches numpy lstsq'
print('\n✅ (X^T X)^-1 X^T y solves least squares in closed form')

## Residual plot

In [ ]:
y_hat_syn = X_syn_b @ w_ols
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_syn, y_hat_syn, alpha=0.6, color='#6366f1')
mn, mx = min(y_syn.min(),y_hat_syn.min()), max(y_syn.max(),y_hat_syn.max())
axes[0].plot([mn,mx],[mn,mx],'--',color='#f59e0b'); axes[0].set_xlabel('True y'); axes[0].set_ylabel('Predicted y'); axes[0].set_title('Predicted vs True')
axes[1].scatter(y_hat_syn, y_syn-y_hat_syn, alpha=0.6, color='#6366f1')
axes[1].axhline(0, color='#f59e0b', linestyle='--'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual'); axes[1].set_title('Residual plot')
plt.tight_layout(); plt.show(); print(f"R² = {r_squared(y_syn, y_hat_syn):.3f}")

### Validate: OLS fits well and its residuals are centered

OLS minimizes squared error, so it achieves a high $R^2$ here, and — because it includes a bias
term — the residuals **sum to zero** (a defining property of the least-squares fit). We confirm
both.

In [ ]:
y_hat = X_syn_b @ w_ols
resid = y_syn - y_hat
print(f'R^2 = {r_squared(y_syn, y_hat):.4f},  residual sum = {resid.sum():.2e}')
assert r_squared(y_syn, y_hat) > 0.9, 'OLS achieves a high R^2 on well-specified data'
assert abs(resid.sum()) < 1e-8, 'with a bias term, OLS residuals sum to zero'
print('\n✅ OLS is the minimum-squared-error fit; its residuals are orthogonal to the design')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **multicollinearity** | $X^\top X$ near-singular -> exploding weights (demo) — use ridge |
| **explicit inverse** | inverting $X^\top X$ is unstable — solve the system instead |
| **$O(d^3)$ cost** | large $d$ favors iterative / gradient methods |
| **outliers** | squared loss is sensitive; consider robust regression |
| **more features than samples** | $X^\top X$ singular — regularize |

Demo: collinear features make OLS weights explode; ridge shrinks them.

In [ ]:
# When OLS breaks: MULTICOLLINEARITY. If two features are nearly identical, X^T X is
# near-singular, so its inverse is huge and the OLS weights EXPLODE (wildly large, sign-flipped,
# unstable). Ridge regression adds lambda*I to make X^T X + lambda*I well-conditioned, shrinking
# the weights back to something sensible. We show the blow-up and the fix.
rng_c = np.random.default_rng(1)
x1 = rng_c.standard_normal(100)
x2 = x1 + 1e-6 * rng_c.standard_normal(100)     # x2 ~ x1: near-perfect collinearity
Xc = np.column_stack([x1, x2])
yc = x1 + 0.1 * rng_c.standard_normal(100)
w_ols_c = ols(Xc, yc)                                            # normal equation -> unstable
w_ridge = np.linalg.solve(Xc.T @ Xc + 1.0 * np.eye(2), Xc.T @ yc)  # ridge -> stable
print(f'||w|| : OLS {np.linalg.norm(w_ols_c):.1f}   ridge {np.linalg.norm(w_ridge):.2f}')
assert np.linalg.norm(w_ols_c) > 10 * np.linalg.norm(w_ridge), 'collinearity blows up OLS weights; ridge shrinks them'
print('\nNear-singular X^T X -> exploding OLS weights. Ridge (add lambda I) restores stability.')

## ✏️ Your turn

**Task:** Generate a dataset where $X^TX$ is nearly singular (two highly correlated features). Observe what happens to the OLS weights, then compare with Ridge regression.

```python
X_corr = np.column_stack([X_syn[:,0], X_syn[:,0] + 0.01*rng.standard_normal(n)])
```

In [ ]:
# TODO(you): fit OLS on X_corr, observe large/unstable weights
# Then fit Ridge: w_ridge = np.linalg.solve(X.T@X + lam*np.eye(d), X.T@y)

In [ ]:
# assert: ridge weights should have smaller L2 norm than OLS weights
# assert np.linalg.norm(w_ridge) < np.linalg.norm(w_ols_corr)

<details><summary>Solution</summary>

```python
lam = 1.0
X_c = add_bias(np.column_stack([X_syn[:,0], X_syn[:,0]+0.01*rng.standard_normal(n)]))
w_ols_c = ols(X_c, y_syn)
w_ridge = np.linalg.solve(X_c.T@X_c + lam*np.eye(X_c.shape[1]), X_c.T@y_syn)
print(f'OLS   |w|={np.linalg.norm(w_ols_c):.2f}')
print(f'Ridge |w|={np.linalg.norm(w_ridge):.2f}')
# Ridge is much smaller — regularization tames the ill-conditioning
```
</details>

## Key takeaways

- **The normal equation** $w^* = (X^\top X)^{-1} X^\top y$ solves OLS in closed form (verified).
- **High $R^2$, centered residuals** — the least-squares fit's defining properties (verified).
- **Solve, don't invert:** use `np.linalg.solve` / `lstsq` for stability.
- **Collinearity blows up OLS** (demo) — ridge ($+\lambda I$) fixes it.